# 03 — Individual-level data availability and NSDUH PUF bridge

This notebook answers two practical questions:

1. What individual-level data are publicly available for YRBSS vs NSDUH?
2. Can we compute a **NSDUH lifetime prescription misuse (ages 12–17)** comparison series from local PUF files?

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)

In [2]:
DATA_ROOT = Path('..') / 'data'
RAW_ROOT = DATA_ROOT / 'raw'
DERIVED_ROOT = DATA_ROOT / 'derived'
OUT_DIR = DERIVED_ROOT / 'yrbss_nsduh'
OUT_DIR.mkdir(parents=True, exist_ok=True)

YR_RAW = RAW_ROOT / 'yrbss'
NSDUH_RAW = RAW_ROOT / 'nsduh'
YR_RAW, NSDUH_RAW

(PosixPath('../data/raw/yrbss'), PosixPath('../data/raw/nsduh'))

## What is available in this environment?

In [3]:
availability = pd.DataFrame([
    {
        'dataset': 'YRBSS aggregated CSV',
        'path': str(YR_RAW / 'yrbss_7xiw-8vpk.csv'),
        'exists': (YR_RAW / 'yrbss_7xiw-8vpk.csv').exists(),
        'notes': 'CDC indicator-style summary data; not individual respondents',
    },
    {
        'dataset': 'YRBSS individual-level files (if manually provided)',
        'path': str(RAW_ROOT / 'yrbss_puf'),
        'exists': (RAW_ROOT / 'yrbss_puf').exists(),
        'notes': 'Expected local folder for full YRBS microdata extracts',
    },
    {
        'dataset': 'NSDUH raw PUF files',
        'path': str(NSDUH_RAW),
        'exists': NSDUH_RAW.exists() and any(NSDUH_RAW.glob('*')),
        'notes': 'Large local files often excluded from GitHub',
    },
    {
        'dataset': 'NSDUH combined derived parquet',
        'path': str(DERIVED_ROOT / 'nsduh_combined_puf.parquet'),
        'exists': (DERIVED_ROOT / 'nsduh_combined_puf.parquet').exists(),
        'notes': 'Output from nsduh_opioid_prevalence/03_puf_analysis.ipynb',
    },
])
availability

,dataset,path,exists,notes
0,YRBSS aggregated CSV,../data/raw/yrbss/yrbss_7xiw-8vpk.csv,True,CDC indicator-style summary data; not individu...
1,YRBSS individual-level files (if manually prov...,../data/raw/yrbss_puf,False,Expected local folder for full YRBS microdata ...
2,NSDUH raw PUF files,../data/raw/nsduh,False,Large local files often excluded from GitHub
3,NSDUH combined derived parquet,../data/derived/nsduh_combined_puf.parquet,False,Output from nsduh_opioid_prevalence/03_puf_ana...


## Attempt NSDUH lifetime prescription misuse extraction (ages 12–17)

The code below supports a **best-effort** PUF bridge:

- If local NSDUH files exist, it tries common variable names for year, age, weights, and lifetime Rx misuse.
- If files are missing, it writes an empty placeholder table so downstream notebooks still run.

> In this cloud environment, large NSDUH PUF files may be unavailable.

In [4]:
def pick_col(cols, candidates):
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_l:
            return cols_l[cand.lower()]
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

def to_binary_ever(s):
    # Conservative mapping for common coding: 1=yes, 2=no
    return s.map(lambda x: 1.0 if x == 1 else (0.0 if x == 2 else np.nan))

In [5]:
out_path = OUT_DIR / 'nsduh_lifetime_rx_misuse_12_17_from_puf.csv'

nsduh_files = []
if NSDUH_RAW.exists():
    nsduh_files = sorted([p for p in NSDUH_RAW.glob('*') if p.suffix.lower() in {'.dta','.csv','.tsv'}])

records = []
if nsduh_files:
    for fp in nsduh_files:
        try:
            if fp.suffix.lower() == '.dta':
                df = pd.read_stata(fp, convert_categoricals=False)
            elif fp.suffix.lower() == '.csv':
                df = pd.read_csv(fp, low_memory=False)
            else:
                df = pd.read_csv(fp, sep='\t', low_memory=False)
        except Exception as e:
            print(f'Failed to read {fp.name}: {e}')
            continue

        year_col = pick_col(df.columns, ['year'])
        age_col = pick_col(df.columns, ['catag6','catag7','age2','age3'])
        wt_col = pick_col(df.columns, ['analwt2_c','analwtq1q4_c','analwt_c'])
        life_rx_col = pick_col(df.columns, ['pnrnmrec','pnrnmevr','pnrnmlif','pnrnm'])

        if not all([year_col, age_col, wt_col, life_rx_col]):
            continue

        tmp = df[[year_col, age_col, wt_col, life_rx_col]].copy()
        tmp.columns = ['year','age','weight','life_rx_raw']

        # Broad youth filter; map common age categories where 1 often denotes 12-17 in CATAG6
        youth = tmp['age'].isin([1,2]) | tmp['age'].between(12,17, inclusive='both')
        tmp = tmp[youth].copy()

        tmp['life_rx'] = to_binary_ever(tmp['life_rx_raw'])
        tmp = tmp.dropna(subset=['year','weight','life_rx'])

        if len(tmp):
            by_year = tmp.groupby('year').apply(lambda g: np.average(g['life_rx'], weights=g['weight']) * 100).reset_index(name='prevalence_pct')
            for _, r in by_year.iterrows():
                records.append({'year': int(r['year']), 'prevalence_pct': float(r['prevalence_pct'])})

if records:
    life = pd.DataFrame(records).groupby('year', as_index=False)['prevalence_pct'].mean()
    life['source'] = 'NSDUH_PUF'
    life['series'] = 'NSDUH: Lifetime Rx misuse (12-17, PUF-derived)'
else:
    life = pd.DataFrame(columns=['year','prevalence_pct','source','series'])

life.to_csv(out_path, index=False)
print('Saved:', out_path)
life.tail()

Saved: ../data/derived/yrbss_nsduh/nsduh_lifetime_rx_misuse_12_17_from_puf.csv


,year,prevalence_pct,source,series


If `life` is empty here, rerun this notebook on your laptop with local NSDUH raw files available under `data/raw/nsduh/` to populate the PUF-derived lifetime series.